In [1]:
from langchain_openai import ChatOpenAI
import os
#from langchain_google_genai import ChatGoogleGenerativeAI

base_url= os.getenv("AICREDITS_BASE_URL")
api_key = os.getenv("AICREDITS_API_KEY")

In [2]:
llm = ChatOpenAI(
    model="google/gemini-2.5-flash",
    api_key=api_key,
    base_url=base_url,
    #temperature=2.0
)

# Test the model
response = llm.invoke("Hello!")
print(response.content)


Hello! How can I help you today?


In [3]:
import numexpr as ne
import math
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Calculates a single mathematical expression, incl. complex numbers.
    Always add * to operations, examples:
    73i -> 73*i
    7pi**2 -> 7*pi**2
    """
    math_constants = {"pi": math.pi, "i": 1j, "e": math.exp}
    result = ne.evaluate(expression.strip(), local_dict=math_constants)
    return str(result)

In [4]:
from langchain_core.tools import BaseTool
assert isinstance(calculator, BaseTool)
print(f"Tool schema: {calculator.args_schema.model_json_schema()}")

Tool schema: {'description': 'Calculates a single mathematical expression, incl. complex numbers.\nAlways add * to operations, examples:\n73i -> 73*i\n7pi**2 -> 7*pi**2', 'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculator', 'type': 'object'}


In [5]:
from langgraph.prebuilt import create_react_agent

query = "how much is 2+3i squared?"

agent = create_react_agent(llm, [calculator])

for event in agent.stream({"messages": [("user", query)]}, stream_mode="values"):
    event["messages"][-1].pretty_print()

C:\Users\Shivachetan Ulavi\AppData\Local\Temp\ipykernel_17580\1839774308.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [calculator])


================================ Human Message =================================

how much is 2+3i squared?
================================== Ai Message ==================================
Tool Calls:
  calculator (call_calculator_0)
 Call ID: call_calculator_0
  Args:
    expression: (2+3*i)**2
================================= Tool Message =================================
Name: calculator

(-5+12j)
================================== Ai Message ==================================

The result of (2 + 3i) squared is -5 + 12i.


In [6]:
math_constants = {"pi": math.pi, "i": 1j, "e": math.exp}
ne.evaluate("(2+3*i)**2", local_dict=math_constants)

array(-5.+12.j)

In [7]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

C:\Users\Shivachetan Ulavi\AppData\Local\Temp\ipykernel_17580\218264798.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [8]:
question = "What is a square root of the current US president’s age multiplied by 132?"

system_hint = "Think step-by-step. Always use search to get the fresh information about events or public facts that can change over time. Now is 2026 and remember president elections in the US recently happened."

agent2 = create_react_agent(
    llm, [calculator, search],
    prompt=system_hint
)

for event in agent2.stream({"messages": [("user", question)]}, stream_mode="values"):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is a square root of the current US president’s age multiplied by 132?


C:\Users\Shivachetan Ulavi\AppData\Local\Temp\ipykernel_17580\4177153659.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent2 = create_react_agent(


================================== Ai Message ==================================
Tool Calls:
  duckduckgo_search (call_duckduckgo_search_0)
 Call ID: call_duckduckgo_search_0
  Args:
    query: current US president 2026
================================= Tool Message =================================
Name: duckduckgo_search

4 days ago - On August 29, 2025, the US Court of Appeals for the Federal Circuit ruled 7-4 that many of the Trump tariffs were invalid. The Appeals Court had ruled that the International Emergency Economic Powers Act (1977), did not grant the broad powers which the Trump administration was claiming. This decision did not affect some specific tariffs, such as steel or aluminum which were increased under other presidential authority. On February 20, 2026... 3 days ago - If a candidate has received an absolute majority of electoral votes for president, currently 270 of 538, that person is declared the winner. Otherwise, the House of Representatives must meet to elect a

In [9]:
print(event["messages"][-1].content)

The current US president is Donald Trump, and he is 80 years old in 2026. The square root of his age multiplied by 132 is 1180.643892119889.


Passing custom arguments schema while when we create **convert_runnable_to_tool**

In [10]:
from langchain_core.runnables import RunnableLambda, RunnableConfig
from langchain_core.tools import tool, convert_runnable_to_tool
from pydantic import BaseModel, Field

class CalculatorArgs(BaseModel):
    expression: str = Field(description="Mathematical expression to be evaluated")

def calculator(state: CalculatorArgs, config: RunnableConfig) -> str:
    #expression = state["expression"]
    expr = state.expression if isinstance(state, CalculatorArgs) else state.get("expression", "")
    math_constants = config["configurable"].get("math_constants", {})
    result = ne.evaluate(expr.strip(), local_dict=math_constants)
    return str(result)

calculator_with_retry = RunnableLambda(calculator).with_retry(
    wait_exponential_jitter=True,
    stop_after_attempt=3,
)

calculator_tool = convert_runnable_to_tool(
    calculator_with_retry,
    name="calculator",
    description=(
        """Calculates a single mathematical expression, incl. complex numbers.
        Always add * to operations, examples:
        73i -> 73*i
        pi**2 -> 7*pi**2
        """
    ),
    args_schema=CalculatorArgs,
    #arg_types={"expression": "str"},
)

In [11]:
assert isinstance(calculator_tool, BaseTool)
print(f"Tool name: {calculator_tool.name}")
print(f"Tool description: {calculator_tool.description}")
print(f"Arfs schema: {calculator_tool.args_schema.model_json_schema()}")

Tool name: calculator
Tool description: Calculates a single mathematical expression, incl. complex numbers.
        Always add * to operations, examples:
        73i -> 73*i
        pi**2 -> 7*pi**2
Arfs schema: {'properties': {'expression': {'description': 'Mathematical expression to be evaluated', 'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'CalculatorArgs', 'type': 'object'}


In [12]:
math_constants = {"pi": math.pi, "i": 1j, "e": math.exp}
#config = {"configurable": {"math_constants": math_constants}}
config = {"configurable": {"math_constants": math_constants}}

#tool_call = llm.invoke("How much is (2+3i)**2", tools=[calculator_tool]).tool_calls[0]
#rint(tool_call)

In [13]:
llm_with_tools = llm.bind_tools([calculator_tool])
response = llm_with_tools.invoke("How much is (2+3i)**2", config=config)

tool_call = response.tool_calls[0]
print(tool_call)

{'name': 'calculator', 'args': {'expression': '(2+3*i)**2'}, 'id': 'call_calculator_0', 'type': 'tool_call'}


In [14]:
calculator_tool.invoke(tool_call["args"], config=config)

'(-5+12j)'

In [19]:
from langchain_core.tools import StructuredTool
import re

llm2 = ChatOpenAI(
    model="anthropic/claude-fable-5",
    api_key=api_key,
    base_url=base_url,
    #temperature=2.0
)

def calculator(expression: str, config: RunnableConfig) -> str:
    """Calculates a mathematical expression (e.g., '(2+3*i)**2'). 
    Always use '*' for multiplication (3*i) and '**' for exponentiation.
    """
    # 1. Retrieve constants passed in stream(..., config=config)
    math_constants = config.get("configurable", {}).get("math_constants", {"i": 1j})
    
    # 2. Auto-fix common LLM output mistakes ('^' -> '**' and '3i' -> '3*i')
    expr = expression.strip().replace("^", "**")
    expr = re.sub(r'(\d+)(\s*i\b)', r'\1*i', expr)
    
    return str(ne.evaluate(expr, local_dict=math_constants))

calculator_tool = StructuredTool.from_function(
    func=calculator,
    handle_tool_error=True
)

agent3 = create_react_agent(llm2, [calculator_tool])

for event in agent3.stream(
    {"messages": [("user", "How much is (2+3i)^2")]}, 
    stream_mode="values", 
    config=config
):
    event["messages"][-1].pretty_print()

C:\Users\Shivachetan Ulavi\AppData\Local\Temp\ipykernel_17580\280153492.py:29: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent3 = create_react_agent(llm2, [calculator_tool])


================================ Human Message =================================

How much is (2+3i)^2
================================== Ai Message ==================================

I'll calculate that for you.
Tool Calls:
  calculator (toolu_01Gx4mGPcEQuKd9ZrHgqBQ4i)
 Call ID: toolu_01Gx4mGPcEQuKd9ZrHgqBQ4i
  Args:
    expression: (2+3*i)**2
================================= Tool Message =================================
Name: calculator

(-5+12j)
================================== Ai Message ==================================

**(2+3i)² = -5 + 12i**

This comes from expanding: (2+3i)² = 4 + 12i + 9i² = 4 + 12i − 9 = **−5 + 12i**
